# cinematic-controlnet pipeline walkthrough

This notebook runs the neural physics solver, generates a flow field, and converts it into diffusion conditioning tensors.

In [ ]:
import torch

from conditioning.flow_conditioner import FlowConditioner
from physics.flow_generator import OpticalFlowGenerator
from physics.neural_continuum_solver import NeuralContinuumSolver

In [ ]:
solver = NeuralContinuumSolver(latent_dim=64, hidden_dim=32, output_h=16, output_w=16)
generator = OpticalFlowGenerator(state_dim=64, base_channels=16, output_h=16, output_w=16)
conditioner = FlowConditioner(hidden_dim=32, conditioning_dim=128)

latent = torch.randn(1, 8, 64)
force = torch.randn(1, 6)
solver_flow, coarse_rgb = solver(latent, force)
physics_state = solver.force_encoder(force)
generated_flow = generator(physics_state, coarse_rgb.permute(0, 3, 1, 2))
conditioning = conditioner(generated_flow.permute(0, 2, 3, 1), coarse_rgb)
conditioning.shape